# nnU-Net Autosegmentation QC

Visual QC for PI-CAI-trained prostate gland and csPCa lesion candidate predictions on external PROSTATE-MRI cases.

Labels: `1 = prostate_gland`, `2 = cspca_lesion_candidate`.

These overlays are model diagnostics, not clinical proof of cancer localization.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import SimpleITK as sitk

CONFIG = Path('../nnunet_autosegmentation/config/picai_gland_lesion_nnunet_config.json')
config = json.loads(CONFIG.read_text())
image_dir = Path('..') / config['feature_extraction']['image_dir']
mask_dir = Path('..') / config['feature_extraction']['mask_dir']
print(image_dir)
print(mask_dir)

In [ ]:
masks = sorted(mask_dir.glob('*.nii.gz'))
print('prediction masks:', len(masks))
for path in masks[:10]:
    print(path.name)

In [ ]:
def load_case(case_id):
    image = sitk.GetArrayFromImage(sitk.ReadImage(str(image_dir / f'{case_id}_0000.nii.gz')))
    mask = sitk.GetArrayFromImage(sitk.ReadImage(str(mask_dir / f'{case_id}.nii.gz')))
    return image, mask

def choose_slice(mask):
    lesion_counts = (mask == 2).sum(axis=(1, 2))
    if lesion_counts.max() > 0:
        return int(lesion_counts.argmax())
    gland_counts = (mask == 1).sum(axis=(1, 2))
    if gland_counts.max() > 0:
        return int(gland_counts.argmax())
    return mask.shape[0] // 2

def show_case(case_id):
    image, mask = load_case(case_id)
    z = choose_slice(mask)
    gland = np.ma.masked_where(mask[z] != 1, mask[z])
    lesion = np.ma.masked_where(mask[z] != 2, mask[z])
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(image[z], cmap='gray')
    ax.imshow(gland, cmap='Greens', alpha=0.35, vmin=0, vmax=2)
    ax.imshow(lesion, cmap='Reds', alpha=0.65, vmin=0, vmax=2)
    ax.set_title(f'{case_id} | slice {z} | gland voxels={(mask == 1).sum()} | lesion voxels={(mask == 2).sum()}')
    ax.axis('off')
    plt.show()

In [ ]:
for mask_path in masks[:5]:
    show_case(mask_path.name.replace('.nii.gz', ''))